# Apple Podcasts Scraping Exploration

This notebook was used to explore CSS selectors, test dynamic page scraping behavior, and prototype the scraping workflow for Apple Podcasts.
The cleaned, reusable implementation was later consolidated into a Python script for portfolio use.

## Setup

In [4]:
import pandas as pd
from selenium import webdriver
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

podcast_url = "https://podcasts.apple.com/us/podcast/more-attention-less-deficit/id312831485"

options = webdriver.ChromeOptions()
options.add_argument("--headless=new")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 10)

## Open target page

In [5]:
driver.get(podcast_url)
print("Opened:", podcast_url)

Opened: https://podcasts.apple.com/us/podcast/more-attention-less-deficit/id312831485


## Load all episodes

In [6]:
while True:
    try:
        load_more_button = wait.until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, 'div[class^="list-button"] button'))
        )
        # JavaScript click is often more reliable on dynamic pages.
        driver.execute_script("arguments[0].click();", load_more_button)
    except TimeoutException:
        # Stop when there is no clickable load-more button.
        break

print("Finished loading all available episodes.")

Finished loading all available episodes.


## Extract episode links

In [7]:
try:
    link_elements = wait.until(
        EC.presence_of_all_elements_located(
            (By.CSS_SELECTOR, 'a[href*="/podcast/"][href*="?i="]')
        )
    )
except TimeoutException:
    link_elements = []

episode_urls = []
seen = set()

for element in link_elements:
    href = element.get_attribute("href")
    if href and "podcasts.apple.com" in href and href not in seen:
        seen.add(href)
        episode_urls.append(href)

print("Collected episodes:", len(episode_urls))
episode_urls[:10]

Collected episodes: 8


['https://podcasts.apple.com/us/podcast/create-good-luck/id312831485?i=1000319074633',
 'https://podcasts.apple.com/us/podcast/fear-of-failure/id312831485?i=1000314567588',
 'https://podcasts.apple.com/us/podcast/interview-with-comedian-dave-kinney/id312831485?i=1000269734136',
 'https://podcasts.apple.com/us/podcast/the-importance-of-paying-attention-to-sex/id312831485?i=1000256576129',
 'https://podcasts.apple.com/us/podcast/the-crucial-difference-between-facts-and-opinions/id312831485?i=1000236728710',
 'https://podcasts.apple.com/us/podcast/awareness-honesty-and-willingness-the-three-keys/id312831485?i=1000160761940',
 'https://podcasts.apple.com/us/podcast/neurology-matters-but-psychology-matters-more/id312831485?i=1000138033214',
 'https://podcasts.apple.com/us/podcast/the-real-lessons-learned-from-the-chadd-conference/id312831485?i=1000124931105']

## Save results

In [8]:
pd.DataFrame({"episode_url": episode_urls}).to_csv("episodes.csv", index=False)
print("Saved episodes.csv")

Saved episodes.csv


## Notes / observations

- The load-more interaction can take a few seconds between clicks depending on network and rendering speed.
- A JavaScript click (`execute_script`) is more stable than a direct `.click()` on this page.
- `episode_urls` is deduplicated to avoid repeated links.

In [9]:
driver.quit()
print("Driver closed.")

Driver closed.


The main reusable version of this project lives in `scrape_podcasts.py`.
This notebook is kept only as a lightweight exploration record.